# Решения: Мемоизация и DP 1D: размен и максимум

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import csv


def find_data(name):
    for path in (Path(name), Path("../../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден")


def load_coin_cases():
    rows = []
    with find_data("coin_change_cases.csv").open(encoding="utf-8") as file:
        for row in csv.DictReader(file):
            rows.append((
                row["case_id"],
                int(row["amount"]),
                [int(value) for value in row["coins"].split()],
                int(row["expected_min_coins"]),
            ))
    return rows


def load_grid():
    with find_data("route_cost_grid_4x5.csv").open(encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        return [[int(value) for value in row] for row in reader]


COIN_CASES = load_coin_cases()
ROUTE_GRID = load_grid()
assert len(COIN_CASES) == 5
assert len(ROUTE_GRID) == 4 and len(ROUTE_GRID[0]) == 5


## Урок. 1. Где рекурсия повторяет работу

Реализуйте число маршрутов по лестнице с шагами 1 и 2. Счётчик покажет цену повторного вычисления одинаковых состояний.

In [ ]:
plain_calls = 0


def ways_plain(n):
    global plain_calls
    plain_calls += 1
    if n == 0:
        return 1
    if n < 0:
        return 0
    return ways_plain(n - 1) + ways_plain(n - 2)


assert ways_plain(5) == 8
assert plain_calls > 10
print("вызовов:", plain_calls)


## Урок. 2. Мемоизация: состояние вычисляется один раз

Передавайте `cache` явно. Ключ — номер ступени, значение — число способов добраться до неё.

In [ ]:
memo_calls = 0


def ways_memo(n, cache):
    global memo_calls
    memo_calls += 1
    if n == 0:
        return 1
    if n < 0:
        return 0
    if n not in cache:
        cache[n] = ways_memo(n - 1, cache) + ways_memo(n - 2, cache)
    return cache[n]


cache = {}
assert ways_memo(10, cache) == 89
assert set(cache) == set(range(1, 11))
print("вычисленных состояний:", len(cache))


## Урок. 3. Сравнение цены двух решений

Запустите обе версии для `n=18`. Запишите вывод: что именно ограничивает число содержательных вычислений в memo-версии?

In [ ]:
plain_calls = 0
memo_calls = 0
plain_answer = ways_plain(18)
memo_answer = ways_memo(18, {})
CALL_NOTE = (
    "Без кэша одно состояние вычисляется много раз из разных ветвей рекурсии. "
    "Мемоизация оставляет по одному содержательному вычислению для каждого n; "
    "повторные обращения только читают сохранённое значение."
)
assert plain_answer == memo_answer == 4181
assert plain_calls > memo_calls * 20
assert len(CALL_NOTE) >= 100
print(plain_calls, memo_calls, CALL_NOTE)


## Урок. 4. Состояние размена

Для суммы `s` будем хранить минимум монет в `dp[s]`. Заполните базовый случай и значение для недостижимого состояния.

In [ ]:
amount = 6
coins = [1, 3, 4]
unreachable = amount + 1
dp = [unreachable] * (amount + 1)
dp[0] = 0
assert len(dp) == 7
assert dp[0] == 0
assert all(value > amount for value in dp[1:])


## Урок. 5. Минимальный размен

Реализуйте переход `dp[s] = min(dp[s-coin] + 1)`. Если сумму набрать нельзя, функция возвращает `-1`.

In [ ]:
def min_coins(amount, coins):
    unreachable = amount + 1
    dp = [unreachable] * (amount + 1)
    dp[0] = 0
    for current in range(1, amount + 1):
        for coin in coins:
            if current >= coin:
                dp[current] = min(dp[current], dp[current - coin] + 1)
    return -1 if dp[amount] == unreachable else dp[amount]


assert min_coins(6, [1, 3, 4]) == 2
assert min_coins(7, [2, 4]) == -1
assert min_coins(0, [2, 5]) == 0


## Урок. 6. Контракт на данных кассы

Прогоните функцию по CSV. Не подгоняйте код под отдельную строку: одна реализация должна пройти все сочетания сумм и номиналов.

In [ ]:
checked = 0
for case_id, amount, coins, expected in COIN_CASES:
    got = min_coins(amount, coins)
    assert got == expected, (case_id, got, expected)
    checked += 1
assert checked == len(COIN_CASES)
print("проверено кейсов:", checked)


## Урок. 7. Максимум без соседних точек

`dp[i]` — лучший доход на первых `i` точках. Для очередной точки нужно выбрать: пропустить её или взять вместе с результатом до соседней.

In [ ]:
def max_safe_gain(values):
    if not values:
        return 0
    dp = [0] * (len(values) + 1)
    dp[1] = values[0]
    for i in range(2, len(values) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + values[i - 1])
    return dp[-1]


assert max_safe_gain([]) == 0
assert max_safe_gain([8]) == 8
assert max_safe_gain([8, 4, 5, 9, 3, 1, 7]) == 24


## Урок. 8. Эксперимент: жадный размен ошибается

Сравните выбор крупнейшей монеты с DP на суммах 1…12 для номиналов `[1, 3, 4]`. Найдите первую сумму, где ответы различаются.

In [ ]:
def greedy_count(amount, coins):
    count = 0
    for coin in sorted(coins, reverse=True):
        count += amount // coin
        amount %= coin
    return count if amount == 0 else -1


first_failure = None
for amount in range(1, 13):
    if greedy_count(amount, [1, 3, 4]) != min_coins(amount, [1, 3, 4]):
        first_failure = amount
        break
assert first_failure is not None
assert 1 <= first_failure <= 12
print("первый контрпример:", first_failure)


## Урок. 9. Самостоятельно: восстановить набор монет

Верните не только минимум, но и список выбранных монет. Порядок не важен; сумма и длина списка должны подтверждать оптимум.

In [ ]:
def coin_plan(amount, coins):
    unreachable = amount + 1
    dp = [unreachable] * (amount + 1)
    previous_coin = [None] * (amount + 1)
    dp[0] = 0
    for current in range(1, amount + 1):
        for coin in coins:
            if current >= coin and dp[current - coin] + 1 < dp[current]:
                dp[current] = dp[current - coin] + 1
                previous_coin[current] = coin
    if dp[amount] == unreachable:
        return []
    result = []
    while amount > 0:
        coin = previous_coin[amount]
        result.append(coin)
        amount -= coin
    return result


plan = coin_plan(11, [1, 5, 7])
assert sum(plan) == 11
assert len(plan) == 3
assert all(coin in [1, 5, 7] for coin in plan)


## ДЗ. A1. Число маршрутов с тремя шагами

Разрешены шаги 1, 2 и 3. Реализуйте табличное DP без рекурсии.

In [ ]:
def count_routes(distance):
    dp = [0] * (distance + 1)
    dp[0] = 1
    for current in range(1, distance + 1):
        for step in (1, 2, 3):
            if current >= step:
                dp[current] += dp[current - step]
    return dp[distance]


assert count_routes(0) == 1
assert count_routes(4) == 7
assert count_routes(6) == 24


## ДЗ. A2. Размен по всем строкам CSV

Повторите `min_coins` самостоятельно и верните список результатов.

In [ ]:
def min_coins_hw(amount, coins):
    unreachable = amount + 1
    dp = [unreachable] * (amount + 1)
    dp[0] = 0
    for current in range(1, amount + 1):
        for coin in coins:
            if current >= coin:
                dp[current] = min(dp[current], dp[current - coin] + 1)
    return -1 if dp[amount] == unreachable else dp[amount]


answers = [min_coins_hw(amount, coins) for _, amount, coins, _ in COIN_CASES]
assert answers == [2, 2, 3, 3, 3]


## ДЗ. A3. План безопасных смен

Верните индексы выбранных несоседних смен с максимальной выручкой.

In [ ]:
def safe_shift_plan(values):
    dp = [0] * (len(values) + 1)
    if values:
        dp[1] = values[0]
    for i in range(2, len(values) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + values[i - 1])
    result = []
    i = len(values)
    while i > 0:
        if dp[i] == dp[i - 1]:
            i -= 1
        else:
            result.append(i - 1)
            i -= 2
    return list(reversed(result))


values = [6, 7, 1, 30, 8, 2, 4]
indices = safe_shift_plan(values)
assert all(b - a > 1 for a, b in zip(indices, indices[1:]))
assert sum(values[i] for i in indices) == 41


## ДЗ. B1. Ограниченный запас монет

Каждый номинал можно использовать не более заданного числа раз. Верните минимум монет или `-1`.

In [ ]:
def bounded_min_coins(amount, coins, limits):
    unreachable = amount + 1
    dp = [unreachable] * (amount + 1)
    dp[0] = 0
    for coin, limit in zip(coins, limits):
        next_dp = dp[:]
        for current in range(amount + 1):
            if dp[current] == unreachable:
                continue
            for count in range(1, limit + 1):
                target = current + count * coin
                if target <= amount:
                    next_dp[target] = min(next_dp[target], dp[current] + count)
        dp = next_dp
    return -1 if dp[amount] == unreachable else dp[amount]


assert bounded_min_coins(8, [1, 3, 4], [2, 1, 1]) == 3
assert bounded_min_coins(9, [1, 3, 4], [1, 1, 1]) == -1
